In [126]:
import pandas as pd
import numpy as np
import sklearn
from datetime import datetime as dt

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import classification_report, accuracy_score

import pickle
import joblib

results = pd.read_csv("../data/results.csv")

print(sklearn.__version__)

1.4.2


In [127]:
results.count()

date          49477
home_team     49477
away_team     49477
home_score    49453
away_score    49453
tournament    49477
city          49477
country       49477
neutral       49477
dtype: int64

In [128]:
results.dtypes

date           object
home_team      object
away_team      object
home_score    float64
away_score    float64
tournament     object
city           object
country        object
neutral          bool
dtype: object

In [129]:
results_modern = results[(results["date"] >= "2000-01-01") & (results["date"] <= "2026-12-31")].copy()
results_modern = results_modern.drop(['city', 'country'], axis=1)


results_modern.loc[results_modern["home_score"] > results_modern["away_score"], "result"] = 1
results_modern.loc[results_modern["home_score"] < results_modern["away_score"], "result"] = 2
results_modern.loc[results_modern["home_score"] == results_modern["away_score"], "result"] = 0






In [130]:
results_modern.sort_values(by=['date'], ascending=False).head(30)

,date,home_team,away_team,home_score,away_score,tournament,neutral,result
49476,2026-06-27,Croatia,Ghana,NaN,NaN,FIFA World Cup,True,NaN
49475,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,True,NaN
49474,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,True,NaN
49473,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,True,NaN
49472,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,True,NaN
49471,2026-06-27,Algeria,Austria,NaN,NaN,FIFA World Cup,True,NaN
49470,2026-06-26,Senegal,Iraq,NaN,NaN,FIFA World Cup,True,NaN
49469,2026-06-26,Norway,France,NaN,NaN,FIFA World Cup,True,NaN
49468,2026-06-26,Uruguay,Spain,NaN,NaN,FIFA World Cup,True,NaN
49467,2026-06-26,Cape Verde,Saudi Arabia,NaN,NaN,FIFA World Cup,True,NaN


In [131]:
def update_matches(team1, team2, score1, score2, date, tournament, neutral, result, df):
    df.loc[
    (df['date'] == date) &
    (df['tournament'] == tournament) &
    (df['neutral'] == neutral) & 
    (df['home_team'] == team1) & 
    (df['away_team'] == team2), 
    ['home_score', 'away_score', 'result']
] = [score1, score2, result]
    
    
    return df

update_matches('Mexico', 'Czech Republic', 3, 0, '2026-06-24', 'FIFA World Cup', False, 1, results_modern)
update_matches('South Africa', 'South Korea', 1, 0, '2026-06-24', 'FIFA World Cup', True, 1, results_modern)
update_matches('Canada', 'Switzerland', 1, 2, '2026-06-24', 'FIFA World Cup', False, 2, results_modern)
update_matches('Bosnia and Herzegovina', 'Qatar', 3, 1, '2026-06-24', 'FIFA World Cup', True, 1, results_modern)
update_matches('Scotland', 'Brazil', 0, 3, '2026-06-24', 'FIFA World Cup', True, 2, results_modern)
update_matches('Morocco', 'Haiti', 4, 2, '2026-06-24', 'FIFA World Cup', True, 1, results_modern)
update_matches('Japan', 'Sweden', 1, 1, '2026-06-25', 'FIFA World Cup', True, 0, results_modern)
update_matches('Ecuador', 'Germany', 2, 1, '2026-06-25', 'FIFA World Cup', True, 1, results_modern)
update_matches('Tunisia', 'Netherlands', 1, 3, '2026-06-25', 'FIFA World Cup', True, 2, results_modern)
update_matches('Paraguay', 'Australia', 0, 0, '2026-06-25', 'FIFA World Cup', True, 0, results_modern)
update_matches('United States', 'Turkey', 2, 3, '2026-06-25', 'FIFA World Cup', False, 2, results_modern)
update_matches('Curaçao', 'Ivory Coast', 0, 2, '2026-06-25', 'FIFA World Cup', True, 2, results_modern)
update_matches('Egypt', 'Iran', 1, 1, '2026-06-26', 'FIFA World Cup', True, 0, results_modern)
update_matches('New Zealand', 'Belgium', 1, 5, '2026-06-26', 'FIFA World Cup', True, 2, results_modern)
update_matches('Cape Verde', 'Saudi Arabia', 0, 0, '2026-06-26', 'FIFA World Cup', True, 0, results_modern)
update_matches('Uruguay', 'Spain', 0, 1, '2026-06-26', 'FIFA World Cup', True, 2, results_modern)
update_matches('Norway', 'France', 1, 4, '2026-06-26', 'FIFA World Cup', True, 2, results_modern)
update_matches('Senegal', 'Iraq', 5, 0, '2026-06-26', 'FIFA World Cup', True, 1, results_modern)


results_modern.sort_values(by=['date'], ascending=False).head(30)

,date,home_team,away_team,home_score,away_score,tournament,neutral,result
49476,2026-06-27,Croatia,Ghana,NaN,NaN,FIFA World Cup,True,NaN
49475,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,True,NaN
49474,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,True,NaN
49473,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,True,NaN
49472,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,True,NaN
49471,2026-06-27,Algeria,Austria,NaN,NaN,FIFA World Cup,True,NaN
49470,2026-06-26,Senegal,Iraq,5.0,0.0,FIFA World Cup,True,1.0
49469,2026-06-26,Norway,France,1.0,4.0,FIFA World Cup,True,2.0
49468,2026-06-26,Uruguay,Spain,0.0,1.0,FIFA World Cup,True,2.0
49467,2026-06-26,Cape Verde,Saudi Arabia,0.0,0.0,FIFA World Cup,True,0.0


In [132]:
results_modern = results_modern.dropna()


results_modern[['home_score', 'away_score']] = results_modern[['home_score', 'away_score']].astype(int)
results_modern['date'] = pd.to_datetime(results_modern['date'])

In [133]:
results_modern = results_modern.sort_values("date").reset_index(drop=True)

def get_team_history(df):
    home = df[['date', 'home_team', 'home_score', 'away_score']].rename(
        columns={'home_team': 'team', 'home_score': 'goals_for', 'away_score': 'goals_against'}
    )
    away = df[['date', 'away_team', 'away_score', 'home_score']].rename(
        columns={'away_team': 'team', 'away_score': 'goals_for', 'home_score': 'goals_against'}
    )
    return pd.concat([home, away]).sort_values(['team', 'date'])

team_history = get_team_history(results_modern)

team_history['roll_gf_5'] = team_history.groupby('team')['goals_for'].transform(lambda x: x.shift(1).rolling(5).mean())
team_history['roll_ga_5'] = team_history.groupby('team')['goals_against'].transform(lambda x: x.shift(1).rolling(5).mean())

results_modern = results_modern.merge(
    team_history[['date', 'team', 'roll_gf_5', 'roll_ga_5']],
    left_on=['date', 'home_team'], right_on=['date', 'team'], how='left'
).rename(columns={'roll_gf_5': 'home_goals_for_5', 'roll_ga_5': 'home_goals_against_5'}).drop(columns=['team'])

results_modern = results_modern.merge(
    team_history[['date', 'team', 'roll_gf_5', 'roll_ga_5']],
    left_on=['date', 'away_team'], right_on=['date', 'team'], how='left'
).rename(columns={'roll_gf_5': 'away_goals_for_5', 'roll_ga_5': 'away_goals_against_5'}).drop(columns=['team'])


In [134]:
results_modern = results_modern.dropna()
results_modern.sort_values(by=['date'], ascending=False).head(30)

,date,home_team,away_team,home_score,away_score,tournament,neutral,result,home_goals_for_5,home_goals_against_5,away_goals_for_5,away_goals_against_5
25430,2026-06-26,Senegal,Iraq,5,0,FIFA World Cup,True,1.0,1.6,2.0,0.6,2.0
25429,2026-06-26,New Zealand,Belgium,1,5,FIFA World Cup,True,2.0,1.4,2.2,1.8,0.4
25428,2026-06-26,Egypt,Iran,1,1,FIFA World Cup,True,0.0,1.2,0.8,2.4,0.6
25427,2026-06-26,Norway,France,1,4,FIFA World Cup,True,2.0,2.2,1.0,2.6,1.0
25426,2026-06-26,Cape Verde,Saudi Arabia,0,0,FIFA World Cup,True,0.0,1.2,1.2,1.0,1.4
25425,2026-06-26,Uruguay,Spain,0,1,FIFA World Cup,True,2.0,1.0,1.8,1.6,0.4
25422,2026-06-25,United States,Turkey,2,3,FIFA World Cup,False,2.0,2.0,1.4,1.4,0.8
25420,2026-06-25,Curaçao,Ivory Coast,0,2,FIFA World Cup,True,2.0,1.4,3.2,1.8,0.6
25421,2026-06-25,Japan,Sweden,1,1,FIFA World Cup,True,0.0,1.8,0.4,2.4,2.6
25419,2026-06-25,Ecuador,Germany,2,1,FIFA World Cup,True,1.0,1.2,0.6,3.4,0.8


In [135]:
def calculate_match_weight(match_date, tournament_type):

    years_old = (dt.today() - match_date).days / 365
    time_weight = np.exp(-0.138 * years_old)

    if tournament_type == "FIFA World Cup":
        tournament_weight = 1.0
    elif tournament_type == "Friendly":
        tournament_weight = 0.3
    else:
        tournament_weight = 0.7

    return time_weight * tournament_weight

weights = results_modern.apply(
    lambda row: calculate_match_weight(row["date"], row["tournament"]),
    axis=1
)

results_modern.sort_values(by=['date'], ascending=False).head(30)


,date,home_team,away_team,home_score,away_score,tournament,neutral,result,home_goals_for_5,home_goals_against_5,away_goals_for_5,away_goals_against_5
25430,2026-06-26,Senegal,Iraq,5,0,FIFA World Cup,True,1.0,1.6,2.0,0.6,2.0
25429,2026-06-26,New Zealand,Belgium,1,5,FIFA World Cup,True,2.0,1.4,2.2,1.8,0.4
25428,2026-06-26,Egypt,Iran,1,1,FIFA World Cup,True,0.0,1.2,0.8,2.4,0.6
25427,2026-06-26,Norway,France,1,4,FIFA World Cup,True,2.0,2.2,1.0,2.6,1.0
25426,2026-06-26,Cape Verde,Saudi Arabia,0,0,FIFA World Cup,True,0.0,1.2,1.2,1.0,1.4
25425,2026-06-26,Uruguay,Spain,0,1,FIFA World Cup,True,2.0,1.0,1.8,1.6,0.4
25422,2026-06-25,United States,Turkey,2,3,FIFA World Cup,False,2.0,2.0,1.4,1.4,0.8
25420,2026-06-25,Curaçao,Ivory Coast,0,2,FIFA World Cup,True,2.0,1.4,3.2,1.8,0.6
25421,2026-06-25,Japan,Sweden,1,1,FIFA World Cup,True,0.0,1.8,0.4,2.4,2.6
25419,2026-06-25,Ecuador,Germany,2,1,FIFA World Cup,True,1.0,1.2,0.6,3.4,0.8


In [136]:
team_encoder = LabelEncoder()

all_teams = pd.concat([results_modern['home_team'], results_modern['away_team']])
team_encoder.fit(all_teams)

joblib.dump(team_encoder, "../app/team_encoder.pkl")

results_modern['home_team_encoded'] = team_encoder.transform(results_modern['home_team'])
results_modern['away_team_encoded'] = team_encoder.transform(results_modern['away_team'])


tournament_encoder = LabelEncoder()
results_modern['tournament_encoded'] = tournament_encoder.fit_transform(results_modern['tournament'])

joblib.dump(tournament_encoder, "../app/tournament_encoder.pkl")

results_modern['neutral'] = results_modern['neutral'].astype(int)


results_proccesed = results_modern[[
    'date', 
    'home_team', 'away_team', 'tournament',
    'home_team_encoded', 'away_team_encoded', 
    'home_goals_for_5', 'home_goals_against_5', 
    'away_goals_for_5', 'away_goals_against_5', 
    'tournament_encoded', 'neutral'
]].copy()

columnas_float = ['home_goals_for_5', 'home_goals_against_5', 'away_goals_for_5', 'away_goals_against_5']
results_proccesed[columnas_float] = results_proccesed[columnas_float].astype(float)

results_proccesed.to_csv("../app/results_proccesed.csv", index=False)

In [137]:
cutoff_date = pd.to_datetime("2024-01-01")


train_df = results_modern[results_modern["date"] < cutoff_date]
test_df = results_modern[results_modern["date"] >= cutoff_date]

features = ['home_team_encoded', 'away_team_encoded', 'tournament_encoded', 'neutral', 'home_goals_for_5', 'home_goals_against_5', 'away_goals_for_5', 'away_goals_against_5']

X_train = train_df[features]
y_train_home = train_df['home_score']
y_train_away = train_df['away_score']

X_test = test_df[features]
y_test_home = test_df['home_score']
y_test_away = test_df['away_score']

In [138]:

model_home = RandomForestRegressor(n_estimators=100, random_state=1905)

model_away = RandomForestRegressor(n_estimators=100, random_state=1905)

X_train_weights = weights.loc[X_train.index]


model_home.fit(X_train, y_train_home, sample_weight=X_train_weights)
model_away.fit(X_train, y_train_away, sample_weight=X_train_weights)

joblib.dump(model_home, "../app/model_home.pkl", compress=3)
joblib.dump(model_away, "../app/model_away.pkl", compress=3)



['../app/model_away.pkl']

In [139]:

pred_home_continuous = model_home.predict(X_test)
pred_away_continuous = model_away.predict(X_test)


pred_home_rounded = np.round(pred_home_continuous).astype(int)
pred_away_rounded = np.round(pred_away_continuous).astype(int)

In [140]:
mae_home = mean_absolute_error(y_test_home, pred_home_continuous)
mae_away = mean_absolute_error(y_test_away, pred_away_continuous)

print(print(f"MAE Goles Local: {mae_home:.2f} goles de diferencia promedio."))
print(f"MAE Goles Visitante: {mae_away:.2f} goles de diferencia promedio.")

MAE Goles Local: 1.18 goles de diferencia promedio.
None
MAE Goles Visitante: 0.94 goles de diferencia promedio.


In [141]:
# Reconstruir el resultado real en el set de test
actual_result = test_df['result']

# Reconstruir el resultado predicho a partir de los goles redondeados
pred_result = np.where(pred_home_rounded > pred_away_rounded, 1,
                       np.where(pred_home_rounded < pred_away_rounded, 2, 0))

# Calcular el Accuracy del Prode (porcentaje de partidos donde acertaste el Ganador/Empate)
prode_accuracy = accuracy_score(actual_result, pred_result)
print(f"Precisión del Prode (Acierto de 1X2): {prode_accuracy * 100:.2f}%")

# Ver el desglose por categoría (Empates vs Victorias)
print("\nReporte de Clasificación del Prode:")
print(classification_report(actual_result, pred_result, target_names=['Empate (0)', 'Gana Local (1)', 'Gana Visitante (2)']))

Precisión del Prode (Acierto de 1X2): 47.46%

Reporte de Clasificación del Prode:
                    precision    recall  f1-score   support

        Empate (0)       0.26      0.39      0.32       621
    Gana Local (1)       0.60      0.64      0.62      1244
Gana Visitante (2)       0.53      0.27      0.35       752

          accuracy                           0.47      2617
         macro avg       0.47      0.43      0.43      2617
      weighted avg       0.50      0.47      0.47      2617



In [142]:
def get_match_features_live(home_team, away_team, tournament_name, is_neutral, results_df, team_enc, tourn_enc, feature_columns):
    today = pd.to_datetime(dt.today().date())
    
    # 1. Filtrar el historial histórico general de cada equipo antes de hoy
    home_history = results_df[
        ((results_df['home_team'] == home_team) | (results_df['away_team'] == home_team)) & 
        (pd.to_datetime(results_df['date']) < today)
    ].sort_values('date').tail(5)
    
    away_history = results_df[
        ((results_df['home_team'] == away_team) | (results_df['away_team'] == away_team)) & 
        (pd.to_datetime(results_df['date']) < today)
    ].sort_values('date').tail(5)
    

    if len(home_history) < 5 or len(away_history) < 5:
        raise ValueError("Uno de los equipos no tiene al menos 5 partidos previos en el dataset para calcular las métricas.")
        
   
    home_gf = []
    home_gc = []
    for _, row in home_history.iterrows():
        if row['home_team'] == home_team:
            home_gf.append(row['home_score'])
            home_gc.append(row['away_score'])
        else:
            home_gf.append(row['away_score'])
            home_gc.append(row['home_score'])
            
    # 3. Calcular goles a favor y en contra de visitante (Away)
    away_gf = []
    away_gc = []
    for _, row in away_history.iterrows():
        if row['home_team'] == away_team:
            away_gf.append(row['home_score'])
            away_gc.append(row['away_score'])
        else:
            away_gf.append(row['away_score'])
            away_gc.append(row['home_score'])
            
    # 4. Codificar variables categóricas
    home_encoded = team_enc.transform([home_team])[0]
    away_encoded = team_enc.transform([away_team])[0]
    tournament_encoded = tourn_enc.transform([tournament_name])[0]
    
    # 5. Estructurar el DataFrame final coincidiendo con X_train
    live_features = pd.DataFrame([{
        'neutral': int(is_neutral),
        'home_goals_for_5': np.mean(home_gf),
        'home_goals_against_5': np.mean(home_gc),
        'away_goals_for_5': np.mean(away_gf),
        'away_goals_against_5': np.mean(away_gc),
        'home_team_encoded': home_encoded,
        'away_team_encoded': away_encoded,
        'tournament_encoded': tournament_encoded
    }])
    
    return live_features[feature_columns]


input_data = get_match_features_live(
    home_team='Netherlands',
    away_team='Tunisia',
    tournament_name='FIFA World Cup',
    is_neutral=1,
    results_df=results_modern,
    team_enc=team_encoder,
    tourn_enc=tournament_encoder,
    feature_columns=features
)

pred_home = model_home.predict(input_data)[0]
pred_away = model_away.predict(input_data)[0]

print(f"Predicción exacta: Netherlands {pred_home:.2f} - {pred_away:.2f} Tunisia")
print(f"PRODE: Netherlands {int(np.round(pred_home))} - {int(np.round(pred_away))} Tunisia")

Predicción exacta: Netherlands 4.99 - 0.67 Tunisia
PRODE: Netherlands 5 - 1 Tunisia


In [143]:
def predict_neutral_match_symmetric(team_A, team_B, tournament_name, results_df, team_enc, tourn_enc, feature_columns, model_home, model_away):
    # Perspectiva 1: Team A es "Home", Team B es "Away"
    features_1 = get_match_features_live(
        home_team=team_A, away_team=team_B, tournament_name=tournament_name,
        is_neutral=1, results_df=results_df, team_enc=team_enc, tourn_enc=tourn_enc,
        feature_columns=feature_columns
    )
    pred_home_1 = model_home.predict(features_1)[0]
    pred_away_1 = model_away.predict(features_1)[0]
    
    # Perspectiva 2: Invertimos los roles. Team B es "Home", Team A es "Away"
    features_2 = get_match_features_live(
        home_team=team_B, away_team=team_A, tournament_name=tournament_name,
        is_neutral=1, results_df=results_df, team_enc=team_enc, tourn_enc=tourn_enc,
        feature_columns=feature_columns
    )
    
    print(f'Features 1:{features_1}')
    print(f'Features 2:{features_2}')
   
    
    
    pred_home_2 = model_home.predict(features_2)[0]
    pred_away_2 = model_away.predict(features_2)[0]
    
    final_goles_team_A = (pred_home_1 + pred_away_2) / 2
    final_goles_team_B = (pred_away_1 + pred_home_2) / 2
    
    prode_A = int(np.round(final_goles_team_A))
    prode_B = int(np.round(final_goles_team_B))
    
    print(f"Predicción Continua: {team_A} {final_goles_team_A:.2f} - {final_goles_team_B:.2f} {team_B}")
    print(f"Marcador PRODE: {team_A} {prode_A} - {prode_B} {team_B}")
    
    return prode_A, prode_B

predict_neutral_match_symmetric(
    team_A='Netherlands',
    team_B='Tunisia',
    tournament_name='FIFA World Cup',
    results_df=results_modern,
    team_enc=team_encoder,
    tourn_enc=tournament_encoder,
    feature_columns=features,
    model_home=model_home,
    model_away=model_away
)

Features 1:   home_team_encoded  away_team_encoded  tournament_encoded  neutral  \
0                172                259                  51        1   

   home_goals_for_5  home_goals_against_5  away_goals_for_5  \
0               2.4                   1.2               0.4   

   away_goals_against_5  
0                   3.6  
Features 2:   home_team_encoded  away_team_encoded  tournament_encoded  neutral  \
0                259                172                  51        1   

   home_goals_for_5  home_goals_against_5  away_goals_for_5  \
0               0.4                   3.6               2.4   

   away_goals_against_5  
0                   1.2  
Predicción Continua: Netherlands 4.38 - 0.78 Tunisia
Marcador PRODE: Netherlands 4 - 1 Tunisia


(4, 1)